# 03 — Preprocessing

Goal: produce three parallel, directly comparable datasets — full (30 features), correlation-filtered (28 features), and Mann-Whitney significant (24 features) — each split into train / val / test at 60 / 20 / 20.

> **3-way split rationale**: the validation set (`val`) is used by `04_modeling.ipynb` to make the final model selection among tuned candidates. The test set (`test`) is reserved exclusively for `05_evaluation.ipynb` — it is never used in any selection decision, ensuring an unbiased estimate of generalisation performance.
>
> Class-imbalance handling (SMOTE, oversampling, undersampling, `class_weight`) is **not** applied here. It is compared as a variable in `04_modeling.ipynb` and applied only inside each CV training fold to avoid leakage.

In [1]:
import pandas as pd
import numpy as np
import json
import os
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

os.makedirs('../data/processed', exist_ok=True)

## 1. Load Data and Feature Sets

In [2]:
raw = pd.read_csv('../data/raw/creditcard.csv')

with open('../data/processed/selected_features.json') as f:
    selected_features = json.load(f)

with open('../data/processed/mannwhitney_features.json') as f:
    mannwhitney_features = json.load(f)

all_features = [f'V{i}' for i in range(1, 29)] + ['Amount', 'Time']

feature_sets = {
    'full':        all_features,         # 30 features
    'selected':    selected_features,    # 28 features (correlation-filtered)
    'mannwhitney': mannwhitney_features, # 24 features (Mann-Whitney significant at α=0.001)
}

for name, features in feature_sets.items():
    print(f'{name}: {len(features)} features')

full: 30 features
selected: 28 features
mannwhitney: 24 features


## 2. Scale, Split (60/20/20), and Export Each Variant

`V1`–`V28` are already PCA-standardized; only `Amount`/`Time` need scaling, and only if they are present in the given feature set.

All three variants use the same `random_state` so the row membership across splits is identical — PR-AUC comparisons across feature sets are not confounded by different train/val/test rows.

In [3]:
for name, features in feature_sets.items():
    df = raw[features + ['Class']].copy()

    scaler = StandardScaler()
    for col in ['Amount', 'Time']:
        if col in df.columns:
            df[f'{col}_scaled'] = scaler.fit_transform(df[[col]])
            df = df.drop(columns=[col])

    X = df.drop(columns=['Class'])
    y = df['Class']

    # 60 / 20 / 20 split: first carve off 20% test, then split remaining 80% into 75/25
    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
    )

    X_train.assign(Class=y_train.values).to_csv(f'../data/processed/train_{name}.csv', index=False)
    X_val.assign(Class=y_val.values).to_csv(f'../data/processed/val_{name}.csv', index=False)
    X_test.assign(Class=y_test.values).to_csv(f'../data/processed/test_{name}.csv', index=False)

    print(f'[{name}] train: {X_train.shape}, val: {X_val.shape}, test: {X_test.shape}')
    print(f'  fraud rate — train: {y_train.mean():.4f}, val: {y_val.mean():.4f}, test: {y_test.mean():.4f}')
    print(f'  Saved: train_{name}.csv, val_{name}.csv, test_{name}.csv')

[full] train: (170883, 30), val: (56962, 30), test: (56962, 30)
  fraud rate — train: 0.0017, val: 0.0017, test: 0.0017
  Saved: train_full.csv, val_full.csv, test_full.csv


[selected] train: (170883, 28), val: (56962, 28), test: (56962, 28)
  fraud rate — train: 0.0017, val: 0.0017, test: 0.0017
  Saved: train_selected.csv, val_selected.csv, test_selected.csv


[mannwhitney] train: (170883, 24), val: (56962, 24), test: (56962, 24)
  fraud rate — train: 0.0017, val: 0.0017, test: 0.0017
  Saved: train_mannwhitney.csv, val_mannwhitney.csv, test_mannwhitney.csv
